In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils import Util
from steps.prepare_pums import get_filename
from steps.remi_controls import _calculate_pums_rates, aggregate_age_groups, _build_occupation_crosswalk, _normalize_occ_text

util = Util('configs_pypyr')

In [2]:
pums_hh = util.get_table('pums_households_prepared')
pums_person = util.get_table('pums_person_prepared')

In [3]:
pums_person = util.get_table("pums_person_prepared")
pums_hh = util.get_table("pums_households_prepared")
remi = util.get_table("regional_controls")
gq_rates, headship_rates, labor_force_participation_rates, hhsz_rates = _calculate_pums_rates(pums_person, pums_hh)

forecast_year = util.get_setting("forecast_year")
category_col = "category" if "category" in remi.columns else "Category"
age_col = category_col
year_col = forecast_year

remi_age = remi.loc[
    remi[age_col].astype(str).str.contains("ages_", na=False),
    ["county_id", age_col, year_col],
].copy()
remi_age[year_col] = remi_age[year_col] * 1000
remi_age = remi_age.rename(columns={age_col: "age_group", year_col: "total_pop"})
remi_age = remi_age.set_index(["county_id", "age_group"])

remi_age["gq"] = remi_age.index.map(gq_rates).fillna(0) * remi_age["total_pop"]
remi_age["hhpop"] = remi_age["total_pop"] - remi_age["gq"]
remi_age["hh"] = remi_age["hhpop"] * remi_age.index.map(headship_rates).fillna(0)
remi_age["labor_force"] = remi_age.index.map(labor_force_participation_rates).fillna(0) * remi_age["hhpop"]

In [4]:
hhsz_rates_wide = hhsz_rates.unstack()
hhsz_rates_wide.columns = [f"hhsz{int(col)}" if pd.notna(col) else col for col in hhsz_rates_wide.columns]
remi_age = remi_age.merge(hhsz_rates_wide, left_index=True, right_index=True, how="left")
for col in hhsz_rates_wide.columns:
    if col.startswith("hhsz"):
        remi_age[col] = remi_age["hh"] * remi_age[col].fillna(0)

In [5]:
remi_age

total_pop            gq          hhpop  \
county_id age_group                                                  
53033     ages_0_4      120367.289923     66.731412  120300.558511   
          ages_5_9      124950.182993     28.182284  124922.000709   
          ages_10_14    128142.020110    215.108446  127926.911664   
          ages_15_19    135120.436287  11739.067926  123381.368361   
          ages_20_24    133033.443807   5830.874408  127202.569399   
...                               ...           ...            ...   
53061     ages_65_69     46805.244671    591.622196   46213.622475   
          ages_70_74     36282.779410    554.731153   35728.048257   
          ages_75_79     25983.452611    772.184296   25211.268315   
          ages_80_84     14772.286401    993.672821   13778.613580   
          ages_85_plus   11836.097479   1395.562530   10440.534950   

                                  hh   labor_force         hhsz1  \
county_id age_group                                                
53033     ages_0_4          0.000000      0.000000      0.000000   
          ages_5_9          0.000000      0.000000      0.000000   
          ages_10_14        0.000000      0.000000      0.000000   
          ages_15_19     2066.127879  34431.858834    699.887297   
          ages_20_24    34897.922259  99946.506197  16296.877197   
...                              ...           ...           ...   
53061     ages_65_69    26362.766382  17587.037139   8550.058920   
          ages_70_74    21630.032289   5974.286417   8708.718767   
          ages_75_79    15685.222238   2472.697579   6828.890503   
          ages_80_84     8512.621936    543.081327   4000.778160   
          ages_85_plus   6582.755448    203.041026   4386.972403   

                               hhsz2        hhsz3        hhsz4        hhsz5  
county_id age_group                                                          
53033     ages_0_4          0.000000     0.000000     0.000000     0.000000  
          ages_5_9          0.000000     0.000000     0.000000     0.000000  
          ages_10_14        0.000000     0.000000     0.000000     0.000000  
          ages_15_19      552.770338   437.023907   267.190359   109.255977  
          ages_20_24    12543.921613  2891.621516  1745.491505  1420.010429  
...                              ...          ...          ...          ...  
53061     ages_65_69    12984.175792  2849.341951  1107.004419   872.185299  
          ages_70_74     9969.471387  1563.333249   734.038192   654.470694  
          ages_75_79     7047.249095  1201.582198   357.424958   250.075483  
          ages_80_84     3496.827147   697.231049   285.769869    32.015711  
          ages_85_plus   2012.954248   124.948324    29.399606    28.480868  

[72 rows x 10 columns]